In [1]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path


def find_repository_root(start=Path.cwd()):
    """Find the repository root from Jupyter's current working directory."""
    for candidate in (start, *start.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "data").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter from inside "
        "NYC_Healthcare_Accessibility."
    )


REPO_ROOT = find_repository_root()
INPUT_DIR = REPO_ROOT / "data" / "processed" / "intermediate" / "ewm_inputs"
OUTPUT_DIR = REPO_ROOT / "data" / "processed" / "intermediate" / "ewm_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("Libraries loaded.")

Libraries loaded.


In [3]:

df = pd.read_csv(INPUT_DIR / "Staten Island! - STATEN_ISLAND_Indicators_6.csv")
df.head(342)

,Unnamed: 0,from_id,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,1,o_360850003001,2868.357494,2352.011,0,42.183333,39.850000,3.600000
1,2,o_360850003002,2436.623921,2193.933,0,38.233333,37.166667,7.550000
2,3,o_360850006001,1562.643427,1177.950,0,21.100000,19.950000,2.816667
3,4,o_360850006002,2249.393271,1349.659,0,26.533333,22.783333,5.083333
4,5,o_360850007001,2785.689668,2462.690,0,42.216667,41.366667,8.433333
...,...,...,...,...,...,...,...,...
337,338,o_360850319012,5972.368056,1672.545,0,47.733333,28.183333,2.700000
338,339,o_360850319021,7705.749364,2167.702,0,50.500000,36.616667,1.033333
339,340,o_360850319022,7597.816364,2059.769,0,48.583333,34.700000,2.950000
340,341,o_360850319023,7007.502947,2107.874,0,47.716667,35.550000,3.816667


In [4]:
print(df.shape)
print(df.columns.tolist())

(342, 8)
['Unnamed: 0', 'from_id', 'total_distance', 'walking_distance', 'transfers', 'travel_time_total', 'walking_time', 'wait_time_total']


In [5]:
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

df["GEOID_TEXT"] = df["from_id"].astype(str).str.replace("o_", "", regex=False)

df[["from_id", "GEOID_TEXT"]].head(343)

,from_id,GEOID_TEXT
0,o_360850003001,360850003001
1,o_360850003002,360850003002
2,o_360850006001,360850006001
3,o_360850006002,360850006002
4,o_360850007001,360850007001
...,...,...
337,o_360850319012,360850319012
338,o_360850319021,360850319021
339,o_360850319022,360850319022
340,o_360850319023,360850319023


In [6]:
benefit_cols = []
cost_cols = ["total_distance",

    "walking_distance",

    "transfers",

    "travel_time_total",

    "walking_time",

    "wait_time_total"]

criteria_cols = benefit_cols + cost_cols

print("Benefit indicators, higher is better:")
print(benefit_cols)

print("\nCost indicators, lower is better:")
print(cost_cols)

print("\nAll criteria:")
print(criteria_cols)

Benefit indicators, higher is better:
[]

Cost indicators, lower is better:
['total_distance', 'walking_distance', 'transfers', 'travel_time_total', 'walking_time', 'wait_time_total']

All criteria:
['total_distance', 'walking_distance', 'transfers', 'travel_time_total', 'walking_time', 'wait_time_total']


In [7]:
min_max_table = pd.DataFrame({
    "min": df[criteria_cols].min(),
    "max": df[criteria_cols].max()
})

print("Min and max for each indicator:")
display(min_max_table)

Min and max for each indicator:


,min,max
total_distance,726.005058,18087.849730
walking_distance,475.444000,5135.908000
transfers,0.000000,2.000000
travel_time_total,10.033333,120.800000
walking_time,8.050000,86.416667
wait_time_total,1.016667,21.566667


In [8]:
# All of our current indicators are cost indicators
# Lower = better, so we use: (max - value) / (max - min)

normalized = pd.DataFrame(index=df.index)

for col in criteria_cols:
    min_val = df[col].min()
    max_val = df[col].max()
    
    normalized[col] = (max_val - df[col]) / (max_val - min_val)

print("Normalized values:")
display(normalized.head())

Normalized values:


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,0.876606,0.597343,1.0,0.709750,0.594215,0.874290
1,0.901473,0.631262,1.0,0.745411,0.628456,0.682076
2,0.951812,0.849263,1.0,0.900090,0.848150,0.912409
3,0.912257,0.812419,1.0,0.851038,0.811995,0.802109
4,0.881367,0.573595,1.0,0.709449,0.574862,0.639092


In [9]:
r_column_sums = normalized[criteria_cols].sum()

print("Step 2 preparation: Sum of each standardized column")
print("These sums go in the denominator for p_ij.")
display(r_column_sums)

Step 2 preparation: Sum of each standardized column
These sums go in the denominator for p_ij.


total_distance       244.650649
walking_distance     251.827793
transfers            287.500000
travel_time_total    248.052362
walking_time         251.549128
wait_time_total      275.038118
dtype: float64

In [10]:
P = normalized[criteria_cols] / r_column_sums

print("Step 2: Probability matrix p_ij")
print("Each standardized value is divided by its column total.")
display(P.head())

Step 2: Probability matrix p_ij
Each standardized value is divided by its column total.


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,0.003583,0.002372,0.003478,0.002861,0.002362,0.003179
1,0.003685,0.002507,0.003478,0.003005,0.002498,0.002480
2,0.003890,0.003372,0.003478,0.003629,0.003372,0.003317
3,0.003729,0.003226,0.003478,0.003431,0.003228,0.002916
4,0.003603,0.002278,0.003478,0.002860,0.002285,0.002324


In [11]:
print("values should add up to one for each column, since they are probabilities.")
display(P.sum())

values should add up to one for each column, since they are probabilities.


total_distance       1.0
walking_distance     1.0
transfers            1.0
travel_time_total    1.0
walking_time         1.0
wait_time_total      1.0
dtype: float64

In [12]:
P_safe = P.replace(0, 1e-12)

print("Step 3 preparation: Replace 0 values so ln(0) does not break")
display(P_safe.head())

Step 3 preparation: Replace 0 values so ln(0) does not break


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,0.003583,0.002372,0.003478,0.002861,0.002362,0.003179
1,0.003685,0.002507,0.003478,0.003005,0.002498,0.002480
2,0.003890,0.003372,0.003478,0.003629,0.003372,0.003317
3,0.003729,0.003226,0.003478,0.003431,0.003228,0.002916
4,0.003603,0.002278,0.003478,0.002860,0.002285,0.002324


In [13]:
ln_P = np.log(P_safe)

print("Step 3: Natural log of p_ij")
display(ln_P.head())

Step 3: Natural log of p_ij


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,-5.631529,-6.044009,-5.661223,-5.856482,-6.048152,-5.751252
1,-5.603557,-5.988779,-5.661223,-5.807460,-5.992128,-5.999524
2,-5.549219,-5.692132,-5.661223,-5.618900,-5.692336,-5.708577
3,-5.591665,-5.736485,-5.661223,-5.674938,-5.735900,-5.837421
4,-5.626112,-6.084578,-5.661223,-5.856906,-6.081264,-6.064617


In [14]:
P_ln_P = P_safe * ln_P

print("Step 4: p_ij times ln(p_ij)")
display(P_ln_P.head(8))

Step 4: p_ij times ln(p_ij)


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,-0.020178,-0.014337,-0.019691,-0.016757,-0.014287,-0.018282
1,-0.020648,-0.015012,-0.019691,-0.017452,-0.014970,-0.014878
2,-0.021589,-0.019196,-0.019691,-0.020389,-0.019193,-0.018938
3,-0.020850,-0.018506,-0.019691,-0.019470,-0.018515,-0.017024
4,-0.020268,-0.013859,-0.019691,-0.016751,-0.013897,-0.014092
5,-0.019868,-0.012246,-0.019691,-0.015643,-0.012272,-0.019436
6,-0.019950,-0.020774,-0.019691,-0.020448,-0.020787,-0.019035
7,-0.019650,-0.019771,-0.019691,-0.019422,-0.019793,-0.018506


In [15]:
p_ln_p_sums = P_ln_P.sum(axis=0)

print("Step 4 preparation: Sum of p_ij * ln(p_ij) for each indicator")
display(p_ln_p_sums)

Step 4 preparation: Sum of p_ij * ln(p_ij) for each indicator


total_distance      -5.792984
walking_distance    -5.796316
transfers           -5.754044
travel_time_total   -5.808336
walking_time        -5.795955
wait_time_total     -5.799192
dtype: float64

In [16]:
# Step 5: Calculate entropy for each indicator

n = len(normalized)
k = 1 / np.log(n)

entropy = -k * p_ln_p_sums

print("Number of rows:", n)
print("k value:", k)
print("Step 5: Entropy for each indicator")
display(entropy)

Number of rows: 342
k value: 0.1713851648431404
Step 5: Entropy for each indicator


total_distance       0.992831
walking_distance     0.993403
transfers            0.986158
travel_time_total    0.995463
walking_time         0.993341
wait_time_total      0.993896
dtype: float64

In [19]:
# df["fare"].value_counts()

In [17]:
# Step 6: Diversity
# Diversity tells us how much useful variation each indicator has

diversity = 1 - entropy

print("Step 6: Diversity for each indicator")
display(diversity)

Step 6: Diversity for each indicator


total_distance       0.007169
walking_distance     0.006597
transfers            0.013842
travel_time_total    0.004537
walking_time         0.006659
wait_time_total      0.006104
dtype: float64

In [18]:
# Step 7: Calculate entropy weights
# Weight = diversity of one indicator / total diversity of all indicators

weights = diversity / diversity.sum()

print("Step 7: Entropy weights for each indicator")
display(weights)

Step 7: Entropy weights for each indicator


total_distance       0.159623
walking_distance     0.146906
transfers            0.308226
travel_time_total    0.101033
walking_time         0.148283
wait_time_total      0.135929
dtype: float64

In [19]:
weights_table = pd.DataFrame({
    "entropy": entropy,
    "diversity": diversity,
    "weight": weights
})

print("Final entropy weight table:")
display(weights_table)

Final entropy weight table:


,entropy,diversity,weight
total_distance,0.992831,0.007169,0.159623
walking_distance,0.993403,0.006597,0.146906
transfers,0.986158,0.013842,0.308226
travel_time_total,0.995463,0.004537,0.101033
walking_time,0.993341,0.006659,0.148283
wait_time_total,0.993896,0.006104,0.135929


In [20]:
display(weights_table.sort_values(by="weight", ascending=False))

,entropy,diversity,weight
transfers,0.986158,0.013842,0.308226
total_distance,0.992831,0.007169,0.159623
walking_time,0.993341,0.006659,0.148283
walking_distance,0.993403,0.006597,0.146906
wait_time_total,0.993896,0.006104,0.135929
travel_time_total,0.995463,0.004537,0.101033


In [21]:
# Step 8: Calculate final EWM accessibility score
# Formula: score for each block group = sum(normalized value * indicator weight)

df["ewm_accessibility_score"] = (normalized[criteria_cols] * weights).sum(axis=1)

print("Step 8: Final EWM accessibility score")
display(df[["from_id", "GEOID_TEXT", "ewm_accessibility_score"]].head(30))

Step 8: Final EWM accessibility score


,from_id,GEOID_TEXT,ewm_accessibility_score
0,o_360850003001,360850003001,0.814567
1,o_360850003002,360850003002,0.806072
2,o_360850006001,360850006001,0.925647
3,o_360850006002,360850006002,0.888610
4,o_360850007001,360850007001,0.776968
5,o_360850007002,360850007002,0.785708
6,o_360850007003,360850007003,0.938189
7,o_360850007004,360850007004,0.909914
8,o_360850008001,360850008001,0.971888
9,o_360850008002,360850008002,0.879072


In [22]:
print("Score summary:")
display(df["ewm_accessibility_score"].describe())

Score summary:


count    342.000000
mean       0.773127
std        0.154003
min        0.037492
25%        0.659971
50%        0.804698
75%        0.892315
max        0.975771
Name: ewm_accessibility_score, dtype: float64

In [23]:
final_results = df[
    ["GEOID_TEXT", "from_id"] + criteria_cols + ["ewm_accessibility_score"]
].copy()

display(final_results.head(20))

,GEOID_TEXT,from_id,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total,ewm_accessibility_score
0,360850003001,o_360850003001,2868.357494,2352.011,0,42.183333,39.850000,3.600000,0.814567
1,360850003002,o_360850003002,2436.623921,2193.933,0,38.233333,37.166667,7.550000,0.806072
2,360850006001,o_360850006001,1562.643427,1177.950,0,21.100000,19.950000,2.816667,0.925647
3,360850006002,o_360850006002,2249.393271,1349.659,0,26.533333,22.783333,5.083333,0.888610
4,360850007001,o_360850007001,2785.689668,2462.690,0,42.216667,41.366667,8.433333,0.776968
5,360850007002,o_360850007002,3152.799668,2829.800,0,48.433333,47.583333,2.216667,0.785708
6,360850007003,o_360850007003,3077.075041,779.124,0,20.750000,13.183333,2.700000,0.938189
7,360850007004,o_360850007004,3351.584394,1033.524,0,26.816667,17.416667,3.333333,0.909914
8,360850008001,o_360850008001,1291.671364,730.498,0,14.033333,12.350000,1.483333,0.971888
9,360850008002,o_360850008002,1768.931067,1438.267,0,25.316667,24.316667,6.500000,0.879072


In [24]:
final_results.to_csv(OUTPUT_DIR / "staten_island_ewm_results.csv", index=False)
weights_table.to_csv(OUTPUT_DIR / "staten_island_ewm_weights.csv", index=True)

print("Saved staten_island_ewm_results.csv")
print("Saved staten_island_ewm_weights.csv")

Saved staten_island_ewm_results.csv
Saved staten_island_ewm_weights.csv
